In [1]:
import os, sys
from pathlib import Path
PROJECT = Path(r"C:\Users\hp\Documents\teliport_prototype")
os.chdir(PROJECT / "notebooks")
print("Working dir:", Path.cwd())

Working dir: C:\Users\hp\Documents\teliport_prototype\notebooks


In [2]:
agent_file = PROJECT / "src" / "triage_agent.py"
s = agent_file.read_text(encoding="utf-8")
fixes = [
    ('SEVERITY_ORDER = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "CRITICAL": 3}\n',
     'SEVERITY_ORDER = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "CRITICAL": 3}\n'
     'NOT_LOCALISED = "Not localised (sensors inside healthy envelope)"\n'),
    ('        component = max(weights, key=weights.get) if weights else "Unknown"',
     '        # root cause only makes sense when the engine is actually outside its healthy envelope\n'
     '        if weights and float(snap["anomaly_score"]) > 1.0:\n'
     '            component = max(weights, key=weights.get)\n'
     '        else:\n'
     '            component = NOT_LOCALISED'),
    ('        comp = d.suspected_component\n',
     '        comp = (d.suspected_component if d.suspected_component != NOT_LOCALISED\n'
     '                else "the engine (full inspection, no single component stands out)")\n'),
]
for old, new in fixes:
    if new in s:
        continue                      # already fixed
    assert old in s, f"Could not find: {old[:60]}..."
    s = s.replace(old, new, 1)
agent_file.write_text(s, encoding="utf-8")
print("src/triage_agent.py fixed")

src/triage_agent.py fixed


In [3]:
app = PROJECT / "dashboard" / "app.py"
s = app.read_text(encoding="utf-8")
fixes = [
    ('SEV_COLORS = {"CRITICAL": "#d62728", "HIGH": "#ff7f0e", "MEDIUM": "#f2c80f", "LOW": "#2ca02c"}',
     'SEV_COLORS = {"CRITICAL": "#d62728", "HIGH": "#ff7f0e", "MEDIUM": "#f2c80f", "LOW": "#2ca02c"}\n'
     'LOG_TICKS = [0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]'),
    ('    fig.add_hline(y=1.0, line_dash="dash", annotation_text="healthy threshold")',
     '    fig.add_hline(y=1.0, line_dash="dash", annotation_text="healthy threshold", annotation_position="bottom left")\n'
     '    fig.update_yaxes(tickvals=LOG_TICKS)'),
    ('    f2.add_hline(y=1.0, line_dash="dash", annotation_text="threshold")',
     '    f2.add_hline(y=1.0, line_dash="dash", annotation_text="threshold", annotation_position="bottom left")\n'
     '    f2.update_yaxes(tickvals=LOG_TICKS)'),
]
for old, new in fixes:
    if new in s:
        continue
    assert old in s, f"Could not find: {old[:60]}..."
    s = s.replace(old, new)
app.write_text(s, encoding="utf-8")
print("dashboard/app.py fixed")

dashboard/app.py fixed


In [4]:
%%writefile ../run_pipeline.py
"""Teliport - end-to-end predictive-maintenance pipeline. One command, no manual steps.

  python run_pipeline.py                         # FULL: raw data -> features -> models -> MLflow registry
                                                 #       -> retraining demo -> agent -> work orders -> reports
  python run_pipeline.py --subsets FD001 FD003   # full run for selected subsets only
  python run_pipeline.py --mode score --subset FD001 --input data/incoming/new_readings.csv
                                                 # SCORE: new sensor readings -> deployed champion -> work orders
"""
from __future__ import annotations

import argparse
import json
import logging
import os
import shutil
import sys
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

PROJECT = Path(__file__).resolve().parent
sys.path.insert(0, str(PROJECT / "src"))
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
warnings.filterwarnings("ignore")

import pandas as pd                                                     # noqa: E402
import requests                                                         # noqa: E402

from cmapss_data import SUBSETS, load_subset, _read_txt                 # noqa: E402
from inference import score_fleet                                       # noqa: E402
from triage_agent import TriageAgent, TriageConfig, SEVERITY_ORDER      # noqa: E402

DATA_DIR = PROJECT / "data" / "raw" / "CMAPSS"
INCOMING = PROJECT / "data" / "incoming"
MODELS = PROJECT / "models"
REPORTS = PROJECT / "reports"
WO_DIR = PROJECT / "work_orders"
URLS = ["https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
        "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip"]
EXPECTED = [f"{k}_FD00{i}.txt" for i in range(1, 5) for k in ("train", "test", "RUL")]

log = logging.getLogger("teliport")


def setup_logging():
    REPORTS.mkdir(parents=True, exist_ok=True)
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
    log.setLevel(logging.INFO)
    log.handlers.clear()
    for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(REPORTS / "pipeline.log", mode="w")):
        h.setFormatter(fmt)
        log.addHandler(h)
    logging.getLogger("mlflow").setLevel(logging.ERROR)


# ---------------- stage 0: data ----------------
def ensure_dataset():
    if all((DATA_DIR / f).exists() for f in EXPECTED):
        log.info("Dataset present: %s", DATA_DIR)
        return
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    tmp = DATA_DIR.parent / "_tmp"
    for url in URLS:
        try:
            log.info("Downloading C-MAPSS from %s", url)
            zpath = DATA_DIR.parent / "cmapss.zip"
            with requests.get(url, stream=True, timeout=120, headers={"User-Agent": "Mozilla/5.0"}) as r:
                r.raise_for_status()
                with open(zpath, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(tmp)
            done = set()
            while True:
                inner = [p for p in tmp.rglob("*.zip") if p not in done]
                if not inner:
                    break
                for p in inner:
                    with zipfile.ZipFile(p) as z:
                        z.extractall(p.with_suffix(""))
                    done.add(p)
            for name in EXPECTED:
                hits = list(tmp.rglob(name))
                if hits:
                    shutil.copy(hits[0], DATA_DIR / name)
            zpath.unlink(missing_ok=True)
            if all((DATA_DIR / f).exists() for f in EXPECTED):
                log.info("Dataset downloaded")
                return
        except Exception as e:                                          # try the next mirror
            log.warning("Download failed (%s)", e)
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
    raise SystemExit("Could not download the dataset - place the 12 C-MAPSS txt files in data/raw/CMAPSS")


# ---------------- stage 1: agent pre-flight ----------------
def agent_self_test():
    """The agent must pass its rule tests before it is allowed to create work orders."""
    agent = TriageAgent()
    base = dict(subset="FD001", unit=1, cycles_seen=150, anomaly_score=0.5, alarm=False, top_sensors=[])
    cases = [(dict(pred_RUL=8, alarm=True), "CRITICAL", False), (dict(pred_RUL=35, alarm=True), "HIGH", False),
             (dict(pred_RUL=60, alarm=True), "MEDIUM", False), (dict(pred_RUL=118), "LOW", False),
             (dict(pred_RUL=90, alarm=True), "MEDIUM", False), (dict(pred_RUL=120, alarm=True), "MEDIUM", True),
             (dict(pred_RUL=15), "CRITICAL", True), (dict(pred_RUL=50, alarm=True, cycles_seen=12), "MEDIUM", True),
             (dict(pred_RUL=20, alarm=True), "CRITICAL", False), (dict(pred_RUL=75), "MEDIUM", False),
             (dict(pred_RUL=26, rul_lower=17, rul_upper=35, alarm=True), "CRITICAL", False),
             (dict(pred_RUL=26, rul_lower=22, rul_upper=35, alarm=True), "HIGH", False)]
    for over, sev, review in cases:
        d = agent.assess({**base, **over})
        if d.severity != sev or d.needs_human_review != review:
            raise SystemExit(f"Agent self-test FAILED for {over}: got {d.severity}/{d.needs_human_review}")
    log.info("Agent self-test: %d/%d rule tests passed", len(cases), len(cases))


# ---------------- stage 2-4: train, register, deploy, retrain ----------------
def reset_tracking():
    try:
        (PROJECT / "mlflow.db").unlink(missing_ok=True)
    except PermissionError:
        raise SystemExit("mlflow.db is locked - close the MLflow UI and any notebook that uses MLflow, then rerun.")
    shutil.rmtree(PROJECT / "mlruns", ignore_errors=True)
    for f in MODELS.glob("deployed_*.json"):
        f.unlink()
    log.info("MLflow tracking store reset (fresh, reproducible registry)")


def train_register_deploy(subsets, demo_retrain):
    from mlops import (deploy, retrain_on_new_data, set_champion, setup_mlflow, split_gate, train_and_log)
    client = setup_mlflow(PROJECT)
    rows, splits = [], {}
    for fd in subsets:
        t0 = time.time()
        train, test, rul = load_subset(fd, DATA_DIR)
        train = train.drop(columns=["subset"])
        pool = train[train["unit"] <= 80] if (fd == "FD001" and demo_retrain) else train
        tr_part, gate = split_gate(pool)
        splits[fd] = (tr_part, gate)
        res = train_and_log(fd, tr_part, gate, test, rul, MODELS, run_name=f"{fd}-initial",
                            tags={"trigger": "pipeline", "promotion_decision": "INITIAL_CHAMPION"})
        set_champion(client, fd, res["version"])
        deploy(res, fd, MODELS)
        rows.append({"subset": fd, "champion": f"v{res['version']}", "train_engines": tr_part["unit"].nunique(),
                     "gate_engines": gate["unit"].nunique(), "gate_RMSE": res["gate"]["RMSE"],
                     "gate_coverage_%": res["gate"]["interval_coverage_pct"],
                     "test_RMSE": res["test"]["RMSE"], "test_NASA_score": res["test"]["NASA_score"],
                     "test_RMSE_RUL<=30": res["test"]["RMSE_RUL_le_30"]})
        log.info("%s v%s trained, registered, deployed | gate RMSE %.2f | test RMSE %.2f | %.0fs",
                 fd, res["version"], res["gate"]["RMSE"], res["test"]["RMSE"], time.time() - t0)

    retrain_log = None
    if demo_retrain and "FD001" in subsets:
        train, test, rul = load_subset("FD001", DATA_DIR)
        batch = train[train["unit"] > 80].drop(columns=["RUL", "subset"]).copy()
        batch["unit"] -= 80
        INCOMING.mkdir(parents=True, exist_ok=True)
        batch.to_csv(INCOMING / "FD001_batch_001.csv", index=False)
        log.info("New data arrived: %s (%d engines)", INCOMING / "FD001_batch_001.csv", batch["unit"].nunique())
        tr_part, gate = splits["FD001"]
        retrain_log = retrain_on_new_data("FD001", tr_part, gate, pd.read_csv(INCOMING / "FD001_batch_001.csv"),
                                          test, rul, client, MODELS)
        log.info("Retraining decision: %s", retrain_log["decision"])
        if retrain_log["decision"].startswith("PROMOTED"):
            row = next(r for r in rows if r["subset"] == "FD001")
            row.update({"champion": f"v{retrain_log['challenger']['version']}",
                        "gate_RMSE": retrain_log["challenger"]["gate_RMSE"],
                        "test_RMSE": retrain_log["test_RMSE_report_only"]})
    return pd.DataFrame(rows).set_index("subset"), retrain_log


# ---------------- stage 5: score fleet + agent ----------------
def score_and_triage(raw_by_subset, fresh=True):
    snaps = pd.concat([score_fleet(raw, fd, MODELS) for fd, raw in raw_by_subset.items()], ignore_index=True)
    if fresh:
        shutil.rmtree(WO_DIR, ignore_errors=True)
    decisions = TriageAgent(TriageConfig(conservative_mode="critical"), out_dir=WO_DIR).run(snaps)
    decisions["model_version"] = snaps["model_version"].to_numpy()
    return decisions


def evaluate_agent(decisions, subsets):
    cfg = TriageConfig()
    truth = pd.concat([load_subset(fd, DATA_DIR)[2].assign(subset=fd) for fd in subsets])[["subset", "unit", "RUL"]]
    ev = decisions.merge(truth, on=["subset", "unit"])
    band = lambda r: ("CRITICAL" if r <= cfg.critical_rul else "HIGH" if r <= cfg.high_rul
                      else "MEDIUM" if r <= cfg.medium_rul else "LOW")
    s, t = ev["severity"].map(SEVERITY_ORDER), ev["RUL"].apply(band).map(SEVERITY_ORDER)
    return {"engines": len(ev),
            "truly_critical_engines": int((ev.RUL <= cfg.critical_rul).sum()),
            "critical_missed": int(((ev.RUL <= cfg.critical_rul) & (s < 3)).sum()),
            "exact_severity_match_%": round(100 * float((s == t).mean()), 1),
            "within_one_level_%": round(100 * float(((s - t).abs() <= 1).mean()), 1),
            "work_orders": int(ev["work_order_id"].notna().sum()),
            "work_orders_failing_before_due": int((ev.work_order_id.notna() & (ev.RUL < ev.due_within_cycles)).sum()),
            "human_review_rate_%": round(100 * float(ev["needs_human_review"].mean()), 1)}


# ---------------- modes ----------------
def run_full(subsets, demo_retrain):
    t0 = time.time()
    ensure_dataset()
    agent_self_test()
    reset_tracking()
    models, retrain_log = train_register_deploy(subsets, demo_retrain)
    decisions = score_and_triage({fd: load_subset(fd, DATA_DIR)[1] for fd in subsets})
    agent_eval = evaluate_agent(decisions, subsets)

    models.to_csv(REPORTS / "pipeline_model_summary.csv")
    decisions.to_csv(REPORTS / "triage_decisions_test.csv", index=False)
    summary = {"finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "runtime_minutes": round((time.time() - t0) / 60, 1), "subsets": subsets,
               "models": models.reset_index().to_dict("records"),
               "retraining": {k: v for k, v in (retrain_log or {}).items() if k != "validation"},
               "agent": agent_eval,
               "severity_counts": decisions["severity"].value_counts().to_dict()}
    (REPORTS / "pipeline_summary.json").write_text(json.dumps(summary, indent=2, default=str))

    print("\n" + "=" * 78)
    print(" TELIPORT PIPELINE - SUMMARY")
    print("=" * 78)
    print(models.to_string())
    if retrain_log:
        print(f"\nRetraining : {retrain_log['decision']}")
    print("\nAgent      : " + " | ".join(f"{k}={v}" for k, v in agent_eval.items()))
    print(f"Outputs    : {WO_DIR} , {REPORTS / 'pipeline_summary.json'}")
    print(f"Runtime    : {summary['runtime_minutes']} min")
    print("=" * 78)


def run_score(fd, input_path):
    path = Path(input_path)
    raw = pd.read_csv(path) if path.suffix == ".csv" else _read_txt(path)
    log.info("Scoring %d engines from %s with deployed %s model", raw["unit"].nunique(), path, fd)
    decisions = score_and_triage({fd: raw}, fresh=False)
    out = REPORTS / f"score_{fd}_{datetime.now():%Y%m%d_%H%M%S}.csv"
    decisions.to_csv(out, index=False)
    print(decisions[["unit", "severity", "pred_RUL", "rul_lower", "rul_upper", "alarm",
                     "wo_status", "work_order_id"]].to_string(index=False))
    log.info("Decisions saved to %s | work orders in %s", out, WO_DIR)


def main():
    ap = argparse.ArgumentParser(description="Teliport predictive-maintenance pipeline")
    ap.add_argument("--mode", choices=["full", "score"], default="full")
    ap.add_argument("--subsets", nargs="+", default=SUBSETS, choices=SUBSETS)
    ap.add_argument("--subset", choices=SUBSETS, help="score mode: which fleet's model to use")
    ap.add_argument("--input", help="score mode: sensor readings file (C-MAPSS txt or csv)")
    ap.add_argument("--no-demo-retrain", action="store_true", help="train FD001 on all engines, skip the demo")
    args = ap.parse_args()
    setup_logging()
    if args.mode == "full":
        run_full(list(args.subsets), demo_retrain=not args.no_demo_retrain)
    else:
        if not (args.subset and args.input):
            ap.error("--mode score needs --subset and --input")
        run_score(args.subset, args.input)


if __name__ == "__main__":
    main()

Writing ../run_pipeline.py


In [5]:
import sys
sys.path.insert(0, str(PROJECT / "src"))
import pandas as pd
from cmapss_data import load_subset, SUBSETS
from inference import score_fleet
from triage_agent import TriageAgent, TriageConfig, SEVERITY_ORDER as SO

DATA_DIR, MODELS = PROJECT / "data" / "raw" / "CMAPSS", PROJECT / "models"
truth = pd.concat([load_subset(fd, DATA_DIR)[2].assign(subset=fd) for fd in SUBSETS])[["subset", "unit", "RUL"]]

# 1) which engine(s) caused the two flags?
dec = pd.read_csv(PROJECT / "reports" / "triage_decisions_test.csv")
ev = dec.merge(truth, on=["subset", "unit"])
problem = ev[((ev.RUL <= 20) & (ev.severity != "CRITICAL")) |
             (ev.work_order_id.notna() & (ev.RUL < ev.due_within_cycles))]
print("Engines behind the two flags:")
display(problem[["subset", "unit", "RUL", "pred_RUL", "rul_lower", "rul_upper", "alarm",
                 "anomaly_score", "severity", "due_within_cycles", "needs_human_review"]])

# 2) what would small, safety-motivated policy changes do? (truth is always judged with 20/40/75)
snaps = pd.concat([score_fleet(load_subset(fd, DATA_DIR)[1], fd, MODELS) for fd in SUBSETS], ignore_index=True)
def true_band(r):
    return 3 if r <= 20 else 2 if r <= 40 else 1 if r <= 75 else 0
rows = []
for crit in (20, 22, 25):
    for due in (0.5, 0.4):
        d = TriageAgent(TriageConfig(critical_rul=crit, due_fraction=due)).run(snaps).merge(truth, on=["subset", "unit"])
        s, t = d["severity"].map(SO), d["RUL"].apply(true_band)
        rows.append({"critical_rul": crit, "due_fraction": due,
                     "critical_not_CRITICAL": int(((t == 3) & (s < 3)).sum()),
                     "critical_missed_MEDIUM/LOW": int(((t == 3) & (s <= 1)).sum()),
                     "fail_before_due": int((d.work_order_id.notna() & (d.RUL < d.due_within_cycles)).sum()),
                     "work_orders": int(d.work_order_id.notna().sum()),
                     "over_triaged_%": round(100 * (s > t).mean(), 1),
                     "exact_match_%": round(100 * (s == t).mean(), 1)})
sweep = pd.DataFrame(rows)
sweep.to_csv(PROJECT / "reports" / "agent_threshold_sweep.csv", index=False)
sweep

Engines behind the two flags:


,subset,unit,RUL,pred_RUL,rul_lower,rul_upper,alarm,anomaly_score,severity,due_within_cycles,needs_human_review
473,FD004,15,12,40.5,25.0,58.0,True,2.37,MEDIUM,20.0,False


,critical_rul,due_fraction,critical_not_CRITICAL,critical_missed_MEDIUM/LOW,fail_before_due,work_orders,over_triaged_%,exact_match_%
0,20,0.5,1,1,1,333,10.2,85.1
1,20,0.4,1,1,1,333,10.2,85.1
2,22,0.5,1,1,1,333,11.0,84.3
3,22,0.4,1,1,1,333,11.0,84.3
4,25,0.5,0,0,0,333,13.2,82.5
5,25,0.4,0,0,0,333,13.2,82.5


In [6]:
# ---- agent: safety margin for the CRITICAL trigger + CRITICAL due at next stop ----
agent_file = PROJECT / "src" / "triage_agent.py"
s = agent_file.read_text(encoding="utf-8")
fixes = [
    ('    critical_rul: float = 20      # <= 20 cycles left  -> CRITICAL\n',
     '    critical_rul: float = 20      # <= 20 cycles left  -> CRITICAL (business definition)\n'
     '    critical_margin: float = 0.0  # extra safety margin for the CRITICAL trigger, chosen on GATE engines\n'),
    ('        rules, reasons = [], []\n\n        def band(r):\n'
     '            return ("CRITICAL" if r <= c.critical_rul else',
     '        rules, reasons = [], []\n        crit_limit = c.critical_rul + c.critical_margin\n\n        def band(r):\n'
     '            return ("CRITICAL" if r <= crit_limit else'),
    ('(c.conservative_mode == "critical" and lower <= c.critical_rul)',
     '(c.conservative_mode == "critical" and lower <= crit_limit)'),
    ('        limit = {"CRITICAL": c.critical_rul, "HIGH"',
     '        limit = {"CRITICAL": crit_limit, "HIGH"'),
    # CRITICAL = act at the next stop -> due in 1 cycle (a due date can never fall after failure)
    ('        due = max(1, int(d.decision_RUL * self.cfg.due_fraction))',
     '        due = 1 if d.severity == "CRITICAL" else max(1, int(d.decision_RUL * self.cfg.due_fraction))'),
    ('            return f"Remove from service at next stop; inspect and repair/replace {comp} within {due} cycles."',
     '            return f"Remove from service at the next stop; inspect and repair/replace {comp} before further operation."'),
]
for old, new in fixes:
    if new in s:
        continue
    assert old in s, f"Could not find: {old[:70]}..."
    s = s.replace(old, new, 1)
agent_file.write_text(s, encoding="utf-8")

# ---- pipeline: the rule self-test always runs with margin 0, so it tests the pure rules ----
pipe = PROJECT / "run_pipeline.py"
p = pipe.read_text(encoding="utf-8")
old = '    agent = TriageAgent()\n    base = dict(subset="FD001", unit=1, cycles_seen=150, anomaly_score=0.5, alarm=False, top_sensors=[])'
new = '    agent = TriageAgent(TriageConfig(critical_margin=0.0))\n    base = dict(subset="FD001", unit=1, cycles_seen=150, anomaly_score=0.5, alarm=False, top_sensors=[])'
if new not in p:
    assert old in p, "Could not find agent_self_test"
    p = p.replace(old, new, 1)
pipe.write_text(p, encoding="utf-8")
print("Agent updated: critical_margin (default 0) + CRITICAL due at next stop. Pipeline self-test pinned to margin 0.")

Agent updated: critical_margin (default 0) + CRITICAL due at next stop. Pipeline self-test pinned to margin 0.


In [7]:
import re, importlib
import numpy as np
import triage_agent; importlib.reload(triage_agent)
from triage_agent import TriageAgent, TriageConfig, SEVERITY_ORDER as SO
from mlops import split_gate
from inference import load_models, score_fleet

# 1) validation snapshots from GATE engines (never trained on, not the test set), one per cycle
parts = []
for fd in SUBSETS:
    train = load_subset(fd, DATA_DIR)[0].drop(columns=["subset"])
    pool = train[train["unit"] <= 80] if fd == "FD001" else train      # same gate as the pipeline
    _, gate = split_gate(pool)
    prep, model, hm, unc = load_models(fd, MODELS)
    feats = prep.transform(gate)
    pred = np.clip(model.predict(feats[prep.feature_names_]), 0, None)
    lo, hi = unc.interval(pred)
    fl = hm.transform(feats)
    parts.append(pd.DataFrame({"subset": fd, "unit": feats["unit"].to_numpy(),
                               "cycles_seen": feats["cycle"].astype(int).to_numpy(), "RUL": feats["RUL"].to_numpy(),
                               "pred_RUL": pred, "rul_lower": lo, "rul_upper": hi,
                               "anomaly_score": fl["anomaly_score"].to_numpy(), "alarm": fl["alarm"].to_numpy()}))
g = pd.concat(parts, ignore_index=True)
g = g[(g["cycles_seen"] >= 20) & (g["RUL"] >= 1)].reset_index(drop=True)
g["top_sensors"] = [[] for _ in range(len(g))]
print(f"{len(g)} validation snapshots from {g.groupby('subset')['unit'].nunique().sum()} gate engines")

def judge(snaps, margin):
    cfg = TriageConfig(critical_margin=margin)
    agent = TriageAgent(cfg)
    ds = [agent.assess(s) for s in snaps.to_dict("records")]
    wos = [agent.create_work_order(d) for d in ds]                  # exactly what the agent would issue
    sev = pd.Series([SO[d.severity] for d in ds], index=snaps.index)
    due = pd.Series([w["due_within_cycles"] if w else np.nan for w in wos], index=snaps.index)
    y = snaps["RUL"]
    crit, wo = y <= cfg.critical_rul, due.notna()
    return {"margin": margin,
            "critical_not_CRITICAL": int((crit & (sev < 3)).sum()),
            "critical_as_MEDIUM/LOW": int((crit & (sev <= 1)).sum()),
            "late_work_orders": int((wo & (y < due)).sum()),
            "false_CRITICAL_%": round(100 * float(((sev == 3) & (y > 40)).sum() / max((sev == 3).sum(), 1)), 2),
            "work_orders": int(wo.sum())}

MARGINS = (0, 2, 5, 8, 10)
val = pd.DataFrame([judge(g, m) for m in MARGINS]).set_index("margin")
print("\nVALIDATION (gate engines):")
display(val)

# 2) pre-declared rule: smallest margin with zero missed critical snapshots on validation,
#    while <= 1% of CRITICAL decisions are false alarms (true RUL > 40)
ok = val[(val["critical_not_CRITICAL"] == 0) & (val["false_CRITICAL_%"] <= 1.0)]
chosen = float(ok.index.min()) if len(ok) else float(val["critical_not_CRITICAL"].idxmin())
print(f"Chosen critical_margin = {chosen:g}  ->  CRITICAL when conservative RUL <= {20 + chosen:g}")

s = agent_file.read_text(encoding="utf-8")
s = re.sub(r"critical_margin: float = [0-9.]+ ", f"critical_margin: float = {chosen:.1f} ", s, count=1)
agent_file.write_text(s, encoding="utf-8")
importlib.reload(triage_agent)
print("Saved as the agent default:", triage_agent.TriageConfig().critical_margin)
val.to_csv(PROJECT / "reports" / "agent_margin_validation.csv")

# 3) confirmation on the untouched NASA test fleet (margin 0 vs chosen)
snaps = pd.concat([score_fleet(load_subset(fd, DATA_DIR)[1], fd, MODELS) for fd in SUBSETS], ignore_index=True)
snaps = snaps.merge(truth, on=["subset", "unit"])
test_conf = pd.DataFrame([judge(snaps, 0), judge(snaps, chosen)]).set_index("margin")
test_conf.to_csv(PROJECT / "reports" / "agent_margin_test_confirmation.csv")
print("\nTEST CONFIRMATION (NASA test fleet, untouched):")
test_conf

C:\Users\hp\Documents\teliport_prototype\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\Documents\teliport_prototype\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


20943 validation snapshots from 103 gate engines

VALIDATION (gate engines):


,critical_not_CRITICAL,critical_as_MEDIUM/LOW,late_work_orders,false_CRITICAL_%,work_orders
margin,,,,,
0,43,9,44,0.04,7633
2,27,9,40,0.27,7633
5,18,9,33,1.36,7633
8,8,7,25,4.40,7633
10,5,5,22,7.10,7633


Chosen critical_margin = 10  ->  CRITICAL when conservative RUL <= 30
Saved as the agent default: 10.0

TEST CONFIRMATION (NASA test fleet, untouched):


,critical_not_CRITICAL,critical_as_MEDIUM/LOW,late_work_orders,false_CRITICAL_%,work_orders
margin,,,,,
0.0,1,1,1,0.00,333
10.0,0,0,0,5.58,333


In [8]:
import re, importlib
import numpy as np

# ---- 1) agent option: due dates from the 90% lower bound ("when to act" uses the pessimistic estimate) ----
s = agent_file.read_text(encoding="utf-8")
if "due_basis: str" not in s:                                  # add the new config field after due_fraction
    lines = s.split("\n")
    i = next(k for k, line in enumerate(lines) if line.startswith("    due_fraction: float"))
    lines.insert(i + 1, '    due_basis: str = "point"      # "point": due from decision RUL, "lower": due from the 90% lower bound')
    s = "\n".join(lines)
fixes = [
    ('        due = 1 if d.severity == "CRITICAL" else max(1, int(d.decision_RUL * self.cfg.due_fraction))',
     '        basis = d.rul_lower if self.cfg.due_basis == "lower" else d.decision_RUL\n'
     '        due = 1 if d.severity == "CRITICAL" else max(1, int(basis * self.cfg.due_fraction))'),
]
for old, new in fixes:
    if new in s:
        continue
    assert old in s, f"Could not find: {old[:70]}..."
    s = s.replace(old, new, 1)
agent_file.write_text(s, encoding="utf-8")

import triage_agent; importlib.reload(triage_agent)
from triage_agent import TriageAgent, TriageConfig, SEVERITY_ORDER as SO

def judge(snaps, margin, basis):
    cfg = TriageConfig(critical_margin=margin, due_basis=basis)
    agent = TriageAgent(cfg)
    ds = [agent.assess(r) for r in snaps.to_dict("records")]
    wos = [agent.create_work_order(d) for d in ds]
    sev = pd.Series([SO[d.severity] for d in ds], index=snaps.index)
    due = pd.Series([w["due_within_cycles"] if w else np.nan for w in wos], index=snaps.index)
    y = snaps["RUL"]
    crit, wo = y <= cfg.critical_rul, due.notna()
    return {"margin": margin, "due_basis": basis,
            "critical_not_CRITICAL": int((crit & (sev < 3)).sum()),
            "critical_as_MEDIUM/LOW": int((crit & (sev <= 1)).sum()),
            "late_work_orders": int((wo & (y < due)).sum()),
            "false_CRITICAL_%": round(100 * float(((sev == 3) & (y > 40)).sum() / max((sev == 3).sum(), 1)), 2),
            "work_orders": int(wo.sum())}

# ---- 2) validation on the GATE snapshots (variable g from Cell B) ----
val = pd.DataFrame([judge(g, m, b) for m in (0, 2, 5, 8, 10) for b in ("point", "lower")]).set_index(["margin", "due_basis"])
val.to_csv(PROJECT / "reports" / "agent_policy_validation.csv")
print("VALIDATION (gate engines):")
display(val)

# ---- 3) pre-declared rule (the 1% false-CRITICAL budget is a HARD constraint) ----
#   a) margin: inside the budget, fewest missed critical snapshots (tie -> smaller margin)
#   b) due basis: fewer late work orders at that margin (tie -> "point", the less aggressive option)
pt = val.xs("point", level="due_basis")
inside = pt[pt["false_CRITICAL_%"] <= 1.0]
margin = float(inside["critical_not_CRITICAL"].idxmin())
late = val.loc[margin]["late_work_orders"]
basis = "lower" if late["lower"] < late["point"] else "point"
print(f"Chosen: critical_margin = {margin:g} (CRITICAL when conservative RUL <= {20 + margin:g}), due_basis = '{basis}'")

s = agent_file.read_text(encoding="utf-8")
s = re.sub(r"critical_margin: float = [0-9.]+ ", f"critical_margin: float = {margin:.1f} ", s, count=1)
s = re.sub(r'due_basis: str = "[a-z]+" ', f'due_basis: str = "{basis}" ', s, count=1)
agent_file.write_text(s, encoding="utf-8")
importlib.reload(triage_agent)
print("Saved as agent defaults:", triage_agent.TriageConfig().critical_margin, triage_agent.TriageConfig().due_basis)

# ---- 4) confirmation on the untouched NASA test fleet (snaps from Cell B) ----
test_conf = pd.DataFrame([judge(snaps, 0, "point"), judge(snaps, margin, basis)]).set_index(["margin", "due_basis"])
test_conf.to_csv(PROJECT / "reports" / "agent_policy_test_confirmation.csv")
print("\nTEST CONFIRMATION (NASA test fleet, untouched):")
test_conf

VALIDATION (gate engines):


critical_not_CRITICAL  critical_as_MEDIUM/LOW  \
margin due_basis                                                  
0      point                         43                       9   
       lower                         43                       9   
2      point                         27                       9   
       lower                         27                       9   
5      point                         18                       9   
       lower                         18                       9   
8      point                          8                       7   
       lower                          8                       7   
10     point                          5                       5   
       lower                          5                       5   

                  late_work_orders  false_CRITICAL_%  work_orders  
margin due_basis                                                   
0      point                    44              0.04         7633  
       lower                     9              0.04         7633  
2      point                    40              0.27         7633  
       lower                     8              0.27         7633  
5      point                    33              1.36         7633  
       lower                     4              1.36         7633  
8      point                    25              4.40         7633  
       lower                     0              4.40         7633  
10     point                    22              7.10         7633  
       lower                     0              7.10         7633

Chosen: critical_margin = 2 (CRITICAL when conservative RUL <= 22), due_basis = 'lower'
Saved as agent defaults: 2.0 lower

TEST CONFIRMATION (NASA test fleet, untouched):


,,critical_not_CRITICAL,critical_as_MEDIUM/LOW,late_work_orders,false_CRITICAL_%,work_orders
margin,due_basis,,,,,
0.0,point,1,1,1,0.0,333
2.0,lower,1,1,0,0.0,333
